In [2]:
import scanpy as sc
import torch
import numpy as np
from cancerfoundation.model.model import CancerFoundation
import json
from torch.nn.attention import sdpa_kernel, SDPBackend

# --- DIAGNOSTIC: Check Head Alignment ---
def check_flash_compatibility(model):
    """
    Mechanistically checks if the model architecture permits Flash Attention.
    """
    # Attempt to locate the Transformer Encoder to inspect heads
    # Note: Structure depends on your specific CancerFoundation implementation
    try:
        # Heuristic: Recursive search for a TransformerEncoderLayer or MultiheadAttention
        for name, module in model.named_modules():
            if isinstance(module, torch.nn.MultiheadAttention):
                d_model = module.embed_dim
                n_heads = module.num_heads
                head_dim = d_model / n_heads
                
                print(f"--- ARCHITECTURE CHECK ({name}) ---")
                print(f"d_model: {d_model}")
                print(f"n_heads: {n_heads}")
                print(f"head_dim: {head_dim}")
                
                if head_dim % 8 != 0:
                    print(f"CRITICAL WARNING: head_dim ({head_dim}) is not a multiple of 8.")
                    print("Flash Attention will NOT run. Memory will scale quadratically.")
                    print("Action: Retrain model with d_model divisible by n_heads * 8.")
                    return False
                else:
                    print("Alignment: OK (Multiple of 8)")
                    return True
                break
    except Exception as e:
        print(f"Could not automatically inspect architecture: {e}")
    return True

# --- MAIN SCRIPT ---

# 1. Load Model
with open("./save/my_init_4gpu_lrx2_2/vocab.json", "r") as f:
    vocab = json.load(f)

model = CancerFoundation.load_from_checkpoint(
    './save/my_init_4gpu_lrx2_2/epoch_epoch=14.ckpt', 
    vocab=vocab, 
    strict=True
)

# Force BFloat16 for Flash Attention 2 (Ampere+)
model.eval()
model.cuda()
model.bfloat16() 

# Run Compatibility Check
is_compatible = check_flash_compatibility(model)

# 2. Load Data
data = sc.read_h5ad("./premade_data/expr.h5ad")
common_genes = list(set(vocab.keys()).intersection(set(data.var.index)))
data = data[:, common_genes].copy()

sc.pp.highly_variable_genes(data, n_top_genes=1200, subset=False, layer=None)
data = data[:, data.var['highly_variable']]

# Prepare Tensors
gene_ids = torch.LongTensor([vocab[g] for g in data.var.index])
count_matrix = data.X if isinstance(data.X, np.ndarray) else data.X.toarray()

# 3. Embed
batch_size = 64
embeddings = []

# STRICT CONTEXT: Force Flash Attention. 
# If this block fails, it means the hardware/shapes are incompatible.
try:
    with torch.no_grad():
        # Enforce Flash Attention Backend. If it fails, raise error immediately.
        # This prevents silent fallback to quadratic memory.
        with sdpa_kernel(SDPBackend.FLASH_ATTENTION): 
            
            for i in range(0, len(data), batch_size):
                
                batch_expr = torch.from_numpy(count_matrix[i:i+batch_size]).cuda().bfloat16()
                batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()
                
                # CLS Token
                cls_id = vocab["<cls>"]
                batch_genes = torch.cat([
                    torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
                    batch_genes
                ], dim=1)
                
                batch_expr = torch.cat([
                    torch.full((batch_expr.shape[0], 1), -2, dtype=torch.bfloat16, device='cuda'),
                    batch_expr
                ], dim=1)
                
                # --- CRITICAL FIX: PASS NONE ---
                # Your logic ensures no padding is needed within the batch.
                # Passing None guarantees the "No Mask" Flash kernel is used.
                padding_mask = None 
                
                emb = model.model.embed(
                    src=batch_genes,
                    values=batch_expr,
                    src_key_padding_mask=padding_mask # Pass None here
                )[0]
                
                cell_emb = emb[:, 0, :].float()
                embeddings.append(cell_emb.cpu().numpy())
                
except RuntimeError as e:
    print("\n!!! FLASH ATTENTION FAILED !!!")
    print("The model attempted to fall back, but we forced Flash Attention to debug.")
    print(f"Error: {e}")
    if not is_compatible:
        print("Root Cause: Your model architecture (head_dim) is likely incompatible with Flash Attention.")

if embeddings:
    data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)
    print(f"Successfully embedded {data.shape[0]} cells using Linear Attention.")

--- ARCHITECTURE CHECK (model._orig_mod.transformer_encoder.layers.0.self_attn.self_attn) ---
d_model: 256
n_heads: 8
head_dim: 32.0
Alignment: OK (Multiple of 8)
Successfully embedded 941 cells using Linear Attention.


/tmp/ipykernel_999715/2760677066.py:124: ImplicitModificationWarning: Setting element `.obsm['CancerFoundation']` of view, initializing view as actual.
  data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)


In [1]:
import scanpy as sc
import torch
import numpy as np
from cancerfoundation.model.model import CancerFoundation
import json
from torch.nn.attention import sdpa_kernel, SDPBackend

# --- HELPER: Strict Attention Backend Probe ---
def probe_active_backend(model, src, values, src_key_padding_mask):
    """
    Strictly probes which backend executes by forcing exclusive contexts.
    Returns the single backend that successfully runs.
    """
    # Order of priority for High Performance
    backends = [
        (SDPBackend.FLASH_ATTENTION, "Flash Attention (v2)"),
        (SDPBackend.EFFICIENT_ATTENTION, "Memory Efficient Attention"),
        (SDPBackend.MATH, "Standard Math (Fallback)")
    ]
    
    for backend, name in backends:
        try:
            # Force ONLY this backend
            with sdpa_kernel(backend):
                _ = model.model.embed(src=src, values=values, src_key_padding_mask=src_key_padding_mask)
            return name # Return the first high-priority one that works
        except RuntimeError:
            continue # This backend is incompatible with current dtype/shape
            
    return "Unknown/Failure"

# --- MAIN ---

# 1. Load Model
with open("./save/my_init_4gpu_lrx2_2/vocab.json", "r") as f:
    vocab = json.load(f)

model = CancerFoundation.load_from_checkpoint(
    './save/my_init_4gpu_lrx2_2/epoch_epoch=14.ckpt', 
    vocab=vocab, 
    strict=True
)

# 2. MECHANISTIC FIX: Cast Model to BFloat16
# RTX 4090 supports this natively. Required for Flash Attention v2.
# model.eval()
model.cuda()
model.bfloat16() 

print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"Model Dtype: {next(model.parameters()).dtype} (Target: torch.bfloat16)")

# 3. Load Data
data = sc.read_h5ad("./premade_data/expr.h5ad")
common_genes = list(set(vocab.keys()).intersection(set(data.var.index)))
data = data[:, common_genes].copy()

sc.pp.highly_variable_genes(data, n_top_genes=2400, subset=False, layer=None)
data = data[:, data.var['highly_variable']]

# Prepare Tensors
gene_ids = torch.LongTensor([vocab[g] for g in data.var.index])
count_matrix = data.X if isinstance(data.X, np.ndarray) else data.X.toarray()

# 4. Embed
batch_size = 16
embeddings = []

with torch.no_grad():
    for i in range(0, len(data), batch_size):
        
        # Prepare Batch (Cast inputs to bfloat16!)
        batch_expr = torch.from_numpy(count_matrix[i:i+batch_size]).cuda().bfloat16() 
        batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()
        
        # Add CLS token
        cls_id = vocab["<cls>"]
        batch_genes = torch.cat([
            torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
            batch_genes
        ], dim=1)
        
        # Pad value must be bfloat16 compatible
        batch_expr = torch.cat([
            torch.full((batch_expr.shape[0], 1), -2, dtype=torch.bfloat16, device='cuda'),
            batch_expr
        ], dim=1)
        
        padding_mask = torch.zeros(batch_genes.shape, dtype=torch.bool, device='cuda')
        
        # --- PROBE (First batch only) ---
        if i == 0:
            active = probe_active_backend(model, batch_genes, batch_expr, padding_mask)
            print(f"--- ACTIVE ATTENTION KERNEL: [{active}] ---")
            if "Math" in active:
                print("WARNING: Still using O(N^2) memory. Check tensor alignment.")
        # --------------------------------

        # Inference
        emb = model.model.embed(
            src=batch_genes,
            values=batch_expr,
            src_key_padding_mask=padding_mask
        )[0]
        
        # Convert back to float32/numpy for storage
        cell_emb = emb[:, 0, :].float()
        embeddings.append(cell_emb.cpu().numpy())

data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)
print(f"Embedded {data.shape[0]} cells.")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: NVIDIA GeForce RTX 4090
Model Dtype: torch.bfloat16 (Target: torch.bfloat16)
--- ACTIVE ATTENTION KERNEL: [Flash Attention (v2)] ---
Embedded 941 cells.


/tmp/ipykernel_991262/823485170.py:110: ImplicitModificationWarning: Setting element `.obsm['CancerFoundation']` of view, initializing view as actual.
  data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)


In [1]:
import scanpy as sc
import torch
import numpy as np
from cancerfoundation.model.model import CancerFoundation
from cancerfoundation.data.data_collator import AnnDataCollator
from cancerfoundation.data.dataset import SingleCellDataset
from torch.utils.data import DataLoader
import json


# Load model
with open("./save/my_init_4gpu_lrx2_2/vocab.json", "r") as f:
    vocab = json.load(f)

model = CancerFoundation.load_from_checkpoint(
    './save/my_init_4gpu_lrx2_2/epoch_epoch=14.ckpt', 
    vocab=vocab, 
    strict=True
)
model.eval()
model.cuda()

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CancerFoundation(
  (criterion): MSE()
  (model): OptimizedModule(
    (_orig_mod): TransformerModule(
      (value_encoder): TheirContinuousValueEncoder(
        (dropout): Dropout(p=0.2, inplace=False)
        (linear1): Linear(in_features=1, out_features=256, bias=True)
        (activation): ReLU()
        (linear2): Linear(in_features=256, out_features=256, bias=True)
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (criterion_conditions): CrossEntropyLoss()
      (criterion): MSE()
      (condition_encoders): ModuleDict(
        (technology): ConditionEncoder(
          (embedding): Embedding(15, 256)
          (enc_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        )
      )
      (transformer_encoder): CFGenerator(
        (layers): ModuleList(
          (0-5): 6 x CFLayer(
            (self_attn): MHA(
              (self_attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=256, o

In [2]:
# Load data and filter to common genes
data = sc.read_h5ad("./premade_data/expr.h5ad")
common_genes = list(set(vocab.keys()).intersection(set(data.var.index)))
data = data[:, common_genes].copy()

sc.pp.highly_variable_genes(
    data, 
    n_top_genes=1200, 
    subset=False, # Do not filter yet, just compute and store in .var
    layer=None # Operates on data.X
)
data = data[:, data.var['highly_variable']]

# Prepare tensors manually
gene_ids = torch.LongTensor([vocab[g] for g in data.var.index])
count_matrix = data.X if isinstance(data.X, np.ndarray) else data.X.toarray()

# Embed in batches
batch_size = 64


In [ ]:
# ... (Previous imports and data loading/preparation code) ...

# Embed in batches
batch_size = 16
embeddings = []

# --- CORRECTED MODIFIED BLOCK FOR KERNEL CHECK ---
# Import the necessary enum
from torch.nn.attention import SDPBackend 
import logging

# Configure logging to see PyTorch's internal decisions
logging.basicConfig(level=logging.DEBUG) 
# Note: You might need to set a specific PyTorch environment variable (e.g., TORCH_LOGS=sdpa) 
# to get detailed kernel dispatch logs, but this configuration might show some info.

with torch.no_grad():
    
    for i in range(0, len(data), batch_size):
        # ... (Tensors preparation code remains the same) ...
        batch_expr = torch.FloatTensor(count_matrix[i:i+batch_size]).cuda()
        batch_genes = gene_ids.unsqueeze(0).expand(batch_expr.shape[0], -1).cuda()
        
        # Add CLS token
        cls_id = vocab["<cls>"]
        batch_genes = torch.cat([
            torch.full((batch_expr.shape[0], 1), cls_id, dtype=torch.long, device='cuda'),
            batch_genes
        ], dim=1)
        batch_expr = torch.cat([
            torch.full((batch_expr.shape[0], 1), -2, device='cuda'),  # pad_value for CLS
            batch_expr
        ], dim=1)
        
        # Create padding mask (False = not padded)
        padding_mask = torch.zeros(batch_genes.shape, dtype=torch.bool, device='cuda')
        
        # Get embeddings from transformer
        emb = model.model.embed(
            src=batch_genes,
            values=batch_expr,
            src_key_padding_mask=padding_mask
        )[0]
        
        # Extract CLS token embedding (cell embedding)
        cell_emb = emb[:, 0, :]
        embeddings.append(cell_emb.cpu().numpy())


# ... (Rest of the code to store in AnnData) ...
data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)
print(f"Embedded {data.shape[0]} cells with dimension {data.obsm['CancerFoundation'].shape[1]}")


⚠️ Note: To definitively check the chosen kernel, you must use the PyTorch Profiler.
Look for operator names like 'flash_attention' or 'aten::scaled_dot_product_attention' (for MATH).
Embedded 941 cells with dimension 256


/tmp/ipykernel_966853/511006170.py:67: ImplicitModificationWarning: Setting element `.obsm['CancerFoundation']` of view, initializing view as actual.
  data.obsm["CancerFoundation"] = np.concatenate(embeddings, axis=0)


In [1]:
import sys
sys.path.insert(0, "../")
from cancerfoundation.model.model import CancerFoundation
import scanpy as sc
import os
import json
from cancerfoundation.loss import LossType
import torch
with open("./save/my_init_4gpu_lrx2_2/vocab.json", "r") as f:
    vocab = json.load(f)


model = CancerFoundation.load_from_checkpoint('./save/my_init_4gpu_lrx2_2/epoch_epoch=14.ckpt', vocab=vocab, strict=True)
model

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CancerFoundation(
  (criterion): MSE()
  (model): OptimizedModule(
    (_orig_mod): TransformerModule(
      (value_encoder): TheirContinuousValueEncoder(
        (dropout): Dropout(p=0.2, inplace=False)
        (linear1): Linear(in_features=1, out_features=256, bias=True)
        (activation): ReLU()
        (linear2): Linear(in_features=256, out_features=256, bias=True)
        (norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
      )
      (criterion_conditions): CrossEntropyLoss()
      (criterion): MSE()
      (condition_encoders): ModuleDict(
        (technology): ConditionEncoder(
          (embedding): Embedding(15, 256)
          (enc_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        )
      )
      (transformer_encoder): CFGenerator(
        (layers): ModuleList(
          (0-5): 6 x CFLayer(
            (self_attn): MHA(
              (self_attn): MultiheadAttention(
                (out_proj): NonDynamicallyQuantizableLinear(in_features=256, o

In [2]:
data = sc.read_h5ad("./premade_data/expr.h5ad")
data

AnnData object with n_obs × n_vars = 941 × 17412
    var: 'hgnc_id'

In [ ]:
common_genes = list(set(model.vocab.keys()).intersection(set(data.var.index)))
common_genes

['S100PBP',
 'MREG',
 'SPHK1',
 'ZNF540',
 'SMOC1',
 'REP15',
 'TPH2',
 'GLT1D1',
 'CABP5',
 'NCAM1',
 'TMPRSS3',
 'JAG1',
 'USP46',
 'ESR2',
 'WWC2',
 'HRCT1',
 'HAS3',
 'PEF1',
 'USP45',
 'CYBA',
 'SMPD2',
 'CWC15',
 'CNEP1R1',
 'PBX4',
 'QKI',
 'WNT10A',
 'GLIPR1',
 'ST8SIA2',
 'AGXT2L1',
 'ZNF676',
 'CTBS',
 'ZSCAN12',
 'BARHL2',
 'TP53BP2',
 'DUSP7',
 'PRSS27',
 'NHLH2',
 'SLC24A1',
 'GJB6',
 'PCDHB5',
 'PRMT7',
 'XRCC2',
 'ZNF785',
 'RBBP9',
 'MBD2',
 'ZNF880',
 'ELMOD3',
 'UPF3B',
 'NDUFA8',
 'IGSF9',
 'SEC62',
 'ZMAT5',
 'LRP3',
 'NR0B1',
 'SLC25A21',
 'TRAF3',
 'PPP1R3B',
 'ARHGAP25',
 'CD200R1',
 'SAP130',
 'HINFP',
 'HRH3',
 'POTEC',
 'MZT1',
 'FGD2',
 'LIMK2',
 'FBXL6',
 'TRIM66',
 'HMGCLL1',
 'MT1G',
 'NSUN5',
 'PPP1R14C',
 'PCDHB1',
 'FAM179B',
 'AGAP4',
 'CRCT1',
 'UBR4',
 'EFNB2',
 'OAS3',
 'NLRP9',
 'GBA2',
 'EXOC3L1',
 'FOXA2',
 'TAS2R40',
 'DHX32',
 'HERC2',
 'WISP2',
 'FAM43B',
 'SLC32A1',
 'NAB1',
 'NKD1',
 'OSBPL8',
 'CCNC',
 'SGK2',
 'MRPS35',
 'EZH2',
 'RALGPS2'

In [22]:
data = data[:, list(common_genes)].copy()
data

AnnData object with n_obs × n_vars = 941 × 15838
    var: 'hgnc_id'

In [3]:
model.embed(data)

TypeError: embedding(): argument 'indices' (position 2) must be Tensor, not Index